In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer


In [2]:
df = pd.read_csv("../data/student-mat.csv", sep=";")
df["pass"] = df["G3"].apply(lambda x: 1 if x >= 10 else 0)
df = df.drop(["G1", "G2", "G3"], axis=1)

X = df.drop("pass", axis=1)
y = df["pass"]

cat_cols = X.select_dtypes(include=["object"]).columns
num_cols = X.select_dtypes(exclude=["object"]).columns


C:\Users\USER\AppData\Local\Temp\ipykernel_11272\4230587756.py:8: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include=["object"]).columns


In [3]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
    ]
)


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train_p = preprocessor.fit_transform(X_train)
X_test_p = preprocessor.transform(X_test)


In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_p, y_train)

y_pred_lr = lr.predict(X_test_p)
print("Logistic Accuracy:", accuracy_score(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))


Logistic Accuracy: 0.7341772151898734
              precision    recall  f1-score   support

           0       0.67      0.44      0.53        27
           1       0.75      0.88      0.81        52

    accuracy                           0.73        79
   macro avg       0.71      0.66      0.67        79
weighted avg       0.72      0.73      0.72        79



In [6]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train_p, y_train)

y_pred_rf = rf.predict(X_test_p)
print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))


Random Forest Accuracy: 0.6708860759493671
              precision    recall  f1-score   support

           0       0.57      0.15      0.24        27
           1       0.68      0.94      0.79        52

    accuracy                           0.67        79
   macro avg       0.63      0.55      0.51        79
weighted avg       0.64      0.67      0.60        79



In [7]:
best_model = lr


In [9]:
import joblib
joblib.dump(best_model, "../model/model.pkl")
joblib.dump(preprocessor, "../model/preprocessor.pkl")


['../model/preprocessor.pkl']

“I compared multiple machine learning models and selected Logistic Regression as the final model based on evaluation metrics.”